In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot-stil
sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = "{:.3f}".format

DB_PATH = Path.cwd().parent / "data" / "football.db"
conn = sqlite3.connect(DB_PATH)

# Last inn alt i en DataFrame – 56k rader er trivielt for pandas
df = pd.read_sql("SELECT * FROM matches", conn, parse_dates=["date"])
print(f"Lastet inn {len(df):,} kamper")
df.head()

In [ ]:
# Skala på dataene
print("Antall kamper per liga og sesong (de siste 5 sesongene):")
recent = df[df["season"] >= "2020-21"]
print(recent.groupby(["season", "league"]).size().unstack(fill_value=0))

In [ ]:
# Andel hjemmeseiere per sesong
home_win_rate = (
    df.assign(home_win=(df["result"] == "H").astype(int))
      .groupby(["season", "league"])["home_win"].mean()
      .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.lineplot(data=home_win_rate, x="season", y="home_win", hue="league", ax=ax, marker="o")
ax.set_title("Hjemmebanefordel over tid")
ax.set_ylabel("Andel hjemmeseiere")
ax.set_xlabel("Sesong")
ax.tick_params(axis="x", rotation=60)
ax.axhline(0.33, color="gray", linestyle="--", alpha=0.5, label="Tilfeldig (1/3)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
df["total_goals"] = df["home_goals"] + df["away_goals"]
df["home_goals"] = df["home_goals"].astype(float)
df["away_goals"] = df["away_goals"].astype(float)

goals_per_season = df.groupby(["season", "league"])[["home_goals", "away_goals", "total_goals"]].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
sns.lineplot(data=goals_per_season, x="season", y="total_goals", hue="league", ax=ax, marker="o")
ax.set_title("Gjennomsnittlig antall mål per kamp")
ax.set_ylabel("Mål per kamp")
ax.tick_params(axis="x", rotation=60)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
score_counts = (
    df.groupby(["home_goals", "away_goals"]).size()
      .reset_index(name="count")
)
# Begrens til kamper med <= 5 mål for hvert lag for lesbarhet
score_counts = score_counts[(score_counts["home_goals"] <= 5) & (score_counts["away_goals"] <= 5)]
pivot = score_counts.pivot(index="away_goals", columns="home_goals", values="count").fillna(0)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax, cbar_kws={"label": "Antall kamper"})
ax.invert_yaxis()
ax.set_title("Fordeling av sluttresultater (alle ligaer, alle sesonger)")
ax.set_xlabel("Hjemmemål")
ax.set_ylabel("Bortemål")
plt.tight_layout()
plt.show()